# Dal file di testo alle coppie domanda-risposta

*Anatomia di un LLM* · capitolo 6, Addestrarlo · script `05-01`, `scripts/05_01_dati_e_batching.py`

Il modello impara una cosa sola: dato un pezzo di testo, indovinare il
carattere che viene dopo. Questo script mostra come dalle ricette
dell'Artusi si ricavino centinaia di migliaia di esercizi di quel tipo, e
perche' non serva nessuno a scrivere le risposte.

Si esegue con `python3 scripts/05_01_dati_e_batching.py`
e dura circa 5 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/05_01_dati_e_batching.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [1]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [2]:
import torch

from llm_anatomy.data import (NOME_CORPUS, carica_testo, get_batch,
                              prepara_split)
from llm_anatomy.utils import (byte_leggibili, numero, prepara, stampa_shape,
                               tabella, titolo, verifica)

prepara()

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


device(type='cpu')

In [3]:
titolo("Le ricette dell'Artusi")
testo = carica_testo()
print(f"  file:       data/{NOME_CORPUS}")
print(f"  caratteri:  {numero(len(testo))}")
print(f"  distinti:   {len(set(testo))}")
print(f"  su disco:   {byte_leggibili(len(testo.encode('utf-8')))}")
print()
print("  prime cinque righe:")
for riga in testo.split("\n")[:5]:
    print(f"    | {riga}")


Le ricette dell'Artusi
----------------------
  file:       data/artusi.txt
  caratteri:  876.670
  distinti:   93
  su disco:   860.6 KiB

  prime cinque righe:
    | BRODI, GELATINA E SUGHI
    | 
    | 1. Brodo
    | 
    | Lo sa il popolo e il comune che per ottenere il brodo buono bisogna


In [4]:
titolo("Novantatre' simboli, e nient'altro")
tok, dati_train, dati_val = prepara_split()
print(f"  dimensione: {tok.vocab_size}")
visibili = "".join(c if c not in "\n\r\t" else " " for c in tok.itos)
print(f"  simboli:    {visibili!r}")
print()
frase = "Prima parola."
codificata = tok.encode(frase)
print(f"  encode({frase!r}) = {codificata}")
print(f"  decode(...)      = {tok.decode(codificata)!r}")
verifica(tok.decode(tok.encode(testo[:5000])) == testo[:5000],
         "encode e decode sono l'una l'inversa dell'altra sul corpus")


Novantatre' simboli, e nient'altro
----------------------------------
  dimensione: 93
  simboli:    "  !'(),-./0123456789:;?ABCDEFGHIKLMNOPQRSTUVWXYZabcdefghiklmnopqrstuvwxyzª«º»ÀÈàâçèéêìîòôùûœ—"

  encode('Prima parola.') = [37, 64, 56, 59, 48, 1, 62, 48, 64, 61, 58, 48, 8]
  decode(...)      = 'Prima parola.'
  [verifica superata] encode e decode sono l'una l'inversa dell'altra sul corpus


In [5]:
# Il taglio e' per posizione e non a caso: se le due meta' fossero mescolate,
# una finestra di contesto potrebbe contenere testo di studio e testo di
# controllo insieme, e il controllo non controllerebbe piu' niente.
titolo("Testo per studiare, testo per controllare")
n_tot = len(dati_train) + len(dati_val)
tabella(
    ["insieme", "token", "quota"],
    [["addestramento", numero(len(dati_train)), f"{100 * len(dati_train) / n_tot:.0f}%"],
     ["validazione", numero(len(dati_val)), f"{100 * len(dati_val) / n_tot:.0f}%"]],
)
print("  tagliare per posizione e non a caso impedisce che una finestra di")
print("  contesto contenga token di entrambi gli insiemi")


Testo per studiare, testo per controllare
-----------------------------------------
insieme          token  quota
-------------  -------  -----
addestramento  789.003    90%
validazione     87.667    10%
  tagliare per posizione e non a caso impedisce che una finestra di
  contesto contenga token di entrambi gli insiemi


In [6]:
titolo("La risposta e' la domanda, spostata di uno")
# Finestre minuscole -- quattro sequenze da otto caratteri -- per poterle
# stampare tutte. Nell'addestramento vero sono sedici da centoventotto.
B, T = 4, 8
gen = torch.Generator().manual_seed(1337)
x, y = get_batch(dati_train, B, T, gen)
stampa_shape("x", x)
stampa_shape("y", y)
print()
print("  prima sequenza del batch:")
print(f"    x = {x[0].tolist()}")
print(f"    y = {y[0].tolist()}")
print(f"    x testo: {tok.decode(x[0])!r}")
print(f"    y testo: {tok.decode(y[0])!r}")
verifica(torch.equal(x[0, 1:], y[0, :-1]),
         "y e' x sfalsato di una posizione")


La risposta e' la domanda, spostata di uno
------------------------------------------
  x                            (4, 8)                 int64
  y                            (4, 8)                 int64

  prima sequenza del batch:
    x = [1, 39, 56, 62, 52, 66, 52, 66]
    y = [39, 56, 62, 52, 66, 52, 66, 52]
    x testo: ' Ripetet'
    y testo: 'Ripetete'
  [verifica superata] y e' x sfalsato di una posizione


In [7]:
titolo("Ogni posizione e' un esercizio")
# Il punto meno ovvio del capitolo: una finestra da otto caratteri non e' un
# esercizio, sono otto esercizi, perche' la maschera causale fa in modo che
# ogni posizione veda solo cio' che sta prima di lei.
print("  ogni posizione della finestra e' un esempio a se':")
for t in range(4):
    contesto = tok.decode(x[0, :t + 1])
    bersaglio = tok.decode([y[0, t]])
    print(f"    contesto {contesto!r:<14} -> bersaglio {bersaglio!r}")
print(f"  un batch {B}x{T} contiene quindi {B * T} coppie contesto-bersaglio")


Ogni posizione e' un esercizio
------------------------------
  ogni posizione della finestra e' un esempio a se':
    contesto ' '            -> bersaglio 'R'
    contesto ' R'           -> bersaglio 'i'
    contesto ' Ri'          -> bersaglio 'p'
    contesto ' Rip'         -> bersaglio 'e'
  un batch 4x8 contiene quindi 32 coppie contesto-bersaglio


In [8]:
titolo("Quante volte rilegge il libro")
righe = []
for passi in (500, 1000, 2500):
    token = passi * 16 * 128
    righe.append([numero(passi), numero(token),
                  f"{token / len(dati_train):.2f}".replace(".", ",")])
tabella(["passi", "token visti", "epoche sul corpus"], righe)
volte = 2500 * 16 * 128 / len(dati_train)
print(f"  con B=16 e T=128, i 2500 passi dell'addestramento rileggono il testo "
      f"{volte:.1f}".replace(".", ","))
print("  volte: abbastanza per imparare l'ortografia e le formule che tornano")
print("  in ogni ricetta, troppo poco per sapere le ricette a memoria")


Quante volte rilegge il libro
-----------------------------
passi  token visti  epoche sul corpus
-----  -----------  -----------------
500      1.024.000               1,30
1.000    2.048.000               2,60
2.500    5.120.000               6,49
  con B=16 e T=128, i 2500 passi dell'addestramento rileggono il testo 6,5
  volte: abbastanza per imparare l'ortografia e le formule che tornano
  in ogni ricetta, troppo poco per sapere le ricette a memoria


## Da provare

cambia il punto di taglio fra studio e controllo e rifai il conto.